# 04. Feature Engineering & Selection
## Intelligent Cyber Threat Detection & Network Traffic Analysis

---
### 1. Cyber Domain Feature Synthesis
Network flows possess domain-specific properties:
1. **Packet/Byte Asymmetry:** Forward vs. Backward volume ratios reveal exfiltration or flood asymmetry.
2. **Control Flag Signatures:** High SYN-to-ACK ratios signal SYN flood attacks or TCP port scans.
3. **Packet Size Dispersion:** Standard deviation to mean ratios reveal deterministic automated scripting.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
from src.data_loader import load_dataset
from src.feature_engineering import (
    compute_domain_network_features,
    identify_highly_correlated_features,
    select_top_features_rf,
)
from src.preprocessing import prepare_train_test_split

df, _ = load_dataset(use_sample_if_empty=True)
print(f"Base dataset shape: {df.shape}")

### 2. Engineering Domain Features

In [ ]:
featured_df = compute_domain_network_features(df)
print(f"Shape with engineered features: {featured_df.shape}")
new_cols = [c for c in featured_df.columns if c not in df.columns]
print(f"Added {len(new_cols)} domain features: {new_cols}")
featured_df[new_cols].head(5)

### 3. Collinear Feature Identification

In [ ]:
corr_mat, to_drop, pairs = identify_highly_correlated_features(featured_df, threshold=0.95)
print(f"Pairs with correlation > 0.95: {len(pairs)}")
for p in pairs[:5]:
    print(f"  {p[0]} <---> {p[1]}: r = {p[2]}")

### 4. Random Forest Feature Selection

In [ ]:
X_tr, X_te, y_tr, y_te, prep, _ = prepare_train_test_split(featured_df, mode="binary")
top_features, top_indices, imp_dict = select_top_features_rf(X_tr, y_tr, prep.feature_names_, top_k=20)

print(f"Top 10 Selected Informative Features:")
for idx, (f, imp) in enumerate(list(imp_dict.items())[:10], 1):
    print(f"  {idx:2d}. {f:<30} Importance: {imp:.4f}")